In [1]:
# Set environment variables — must run this FIRST before starting Spark
import os

# Java 11 (compatible with PySpark 3.5.x)
os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-11.0.28.6-hotspot'
os.environ['PATH'] = r'C:\Program Files\Eclipse Adoptium\jdk-11.0.28.6-hotspot\bin' + ';' + os.environ['PATH']

# Hadoop winutils — required for PySpark to work on Windows
os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['PATH'] = r'C:\hadoop\bin' + ';' + os.environ['PATH']

# Tell Spark's JVM which Python executable to use for worker processes
# On Windows the binary is python.exe not python3
os.environ['PYSPARK_PYTHON'] = r'C:\Users\advai\anaconda3\envs\pyspark_env\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = r'C:\Users\advai\anaconda3\envs\pyspark_env\python.exe'

# Remove any external SPARK_HOME so pyspark uses its own bundled JARs (3.5.3)
os.environ.pop('SPARK_HOME', None)

import pyspark
print('JAVA_HOME:', os.environ['JAVA_HOME'])
print('HADOOP_HOME:', os.environ['HADOOP_HOME'])
print('PYSPARK_PYTHON:', os.environ['PYSPARK_PYTHON'])

JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-11.0.28.6-hotspot
HADOOP_HOME: C:\hadoop
PYSPARK_PYTHON: C:\Users\advai\anaconda3\envs\pyspark_env\python.exe


In [2]:
import sys
print(sys.executable)

C:\Users\advai\anaconda3\envs\pyspark_env\python.exe


# Framework Local Testing Notebook

Tests the pure utility functions in `framework.py` using a local PySpark session.
No cloud credentials, Databricks, or Snowflake required.

**Functions covered:**
- Global Utilities: column ops, type casting, MD5 checksums, DML helpers
- Informatica-style Functions: router, joiner, checksum, insert/update/delete flags, lookup
- Regression Testing: schema comparison, hash comparison, dataframe diff, special character check

**Setup:**
```bash
pip install pyspark pandas
```

## 1. Start a Local Spark Session

In [3]:
# display() helper — mimics Databricks display(), renders as HTML table in Jupyter
def display(df, n=20):
    """Show a PySpark DataFrame as a formatted HTML table (like Databricks display)."""
    return df.limit(n).toPandas()

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("framework-local-test") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")  # suppress INFO/WARN noise
print(f"Spark version: {spark.version}")

Spark version: 3.5.3


## 2. Import Framework

The module-level `SparkSession` and `dbutils` lines are commented out in `framework.py`,
so the import will succeed locally. Functions that reference `spark` or `dbutils` at
call-time will use the session created above once we inject it.

In [5]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))  # ensure framework.py is on the path

import importlib
import framework

# Inject the local spark session so framework functions can use it
framework.spark = spark

# Expose all public names into this namespace for convenience
from framework import (
    GetValueFromDataframe, GetTime, julian_to_timestamp, epoch_to_datetime,
    col_rename, truncate_dataframe, column_clear, uppercase_columns,
    trim_column_values, find_and_replace, column_retitle, df_column_rename,
    add_column_prefix, add_prefix_suffix, to_string_datatype,
    generate_update_setString, remove_null_from_dictionary,
    find_and_replace_within_values, dml_operation_df, calculate_df_size,
    infa_router, infa_joiner, md5_checksum,
    generate_insert_update_delete_flags, generate_insert_update_delete_dataframes,
    lookup_dataframe, compare_schemas, hash_comp, compare_dataframes,
    check_for_special_characters, generate_delta_merge_conditions
)
print("Import successful.")

Import successful.


## 3. Sample DataFrames

Reusable test DataFrames used throughout the notebook.

In [6]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Source DataFrame — simulates a raw ingestion payload
source_data = [
    Row(id=1, first_name="Alice",  last_name="Smith",  salary=50000.0, dept="Engineering"),
    Row(id=2, first_name="Bob",    last_name="Jones",  salary=60000.0, dept="Finance"),
    Row(id=3, first_name="Carol",  last_name="White",  salary=55000.0, dept="Engineering"),
    Row(id=4, first_name="  Dave", last_name="Brown ", salary=None,    dept="HR"),
    Row(id=5, first_name="Eve",    last_name="Davis",  salary=70000.0, dept="Finance"),
]

source_schema = StructType([
    StructField("id",         IntegerType(), True),
    StructField("first_name", StringType(),  True),
    StructField("last_name",  StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("dept",       StringType(),  True),
])

source_df = spark.createDataFrame(source_data, schema=source_schema)

# Target DataFrame — simulates existing data in the warehouse
target_data = [
    Row(id=1, first_name="Alice", last_name="Smith", salary=48000.0, dept="Engineering"),  # salary changed
    Row(id=2, first_name="Bob",   last_name="Jones", salary=60000.0, dept="Finance"),      # unchanged
    Row(id=6, first_name="Frank", last_name="Green", salary=45000.0, dept="Ops"),          # delete candidate
]
target_df = spark.createDataFrame(target_data, schema=source_schema)

print("Source DataFrame:")
display(source_df)
print("Target DataFrame:")
display(target_df)

Source DataFrame:
Target DataFrame:


,id,first_name,last_name,salary,dept
0,1,Alice,Smith,48000.0,Engineering
1,2,Bob,Jones,60000.0,Finance
2,6,Frank,Green,45000.0,Ops


## 4. Global Utilities

### 4.1 GetTime

In [7]:
print("Current timestamp (default):", GetTime())
print("Current timestamp (custom fmt):", GetTime(strform='%Y-%m-%d %H:%M:%S'))

Current timestamp (default): 21:33:38
Current timestamp (custom fmt): 2026-03-15 21:33:38


### 4.2 col_rename — rename column dictionary

In [ ]:
# col_rename: regex-based — strips special characters from column names (default)
# Use df_column_rename for explicit old→new mapping
from pyspark.sql.functions import lit
demo_df = source_df.withColumn("full name", lit("test")).withColumn("dept#code", lit("X"))
print("Before:", demo_df.columns)
cleaned_df = col_rename(demo_df)  # strips spaces and # by default
print("After: ", cleaned_df.columns)

# df_column_rename: explicit old→new list
renamed_df = df_column_rename(
    source_df,
    old_list=['first_name', 'last_name', 'salary'],
    new_list=['FirstName', 'LastName', 'AnnualSalary']
)
display(renamed_df)

### 4.3 uppercase_columns

In [ ]:
# uppercase_columns: uppercases ALL column names, no column list argument
upper_df = uppercase_columns(source_df)
print(upper_df.columns)

### 4.4 trim_column_values — strip leading/trailing whitespace

In [ ]:
# trim_column_values: truncates values in ONE column to max_length characters
trimmed_df = trim_column_values(source_df, column_name='first_name', max_length=3)
display(trimmed_df)

### 4.5 add_column_prefix / add_prefix_suffix

In [ ]:
# add_column_prefix: adds prefix to ALL column names, no column selection argument
prefixed_df = add_column_prefix(source_df, prefix='src_')
print(prefixed_df.columns)

### 4.6 to_string_datatype — cast all columns to StringType

In [ ]:
str_df = to_string_datatype(source_df)
print(str_df.schema.simpleString())
display(str_df)

### 4.7 calculate_df_size

In [ ]:
size_bytes = calculate_df_size(source_df)
print(f"Estimated DataFrame size: {size_bytes} bytes")

### 4.8 GetValueFromDataframe

In [ ]:
# Returns first value of a column from the first row
val = GetValueFromDataframe(source_df, 'first_name')
print("First value of first_name:", val)

### 4.9 find_and_replace

In [ ]:
# find_and_replace: replaces a value across the whole DataFrame (no column arg)
replaced_df = find_and_replace(source_df, 'Engineering', 'Tech')
display(replaced_df)

### 4.10 truncate_dataframe — limit to N rows

In [ ]:
# truncate_dataframe: returns an EMPTY DataFrame with the same schema (not a row limit)
empty_df = truncate_dataframe(source_df)
print("Row count after truncate:", empty_df.count())   # 0
print("Schema preserved:", empty_df.schema == source_df.schema)

### 4.11 generate_update_setString

In [ ]:
# generate_update_setString: takes two lists (all columns + PK keys), not a DataFrame
set_str = generate_update_setString(
    column_list=['id', 'first_name', 'last_name', 'salary', 'dept'],
    update_query_keys=['id']
)
print("UPDATE SET clause:")
print(set_str)

### 4.12 generate_delta_merge_conditions

In [ ]:
# generate_delta_merge_conditions: takes two lists (all columns + PK keys), not a DataFrame
merge_result = generate_delta_merge_conditions(
    column_list=['id', 'first_name', 'last_name', 'salary', 'dept'],
    update_query_keys=['id']
)
print("MERGE ON condition:", merge_result[0])
print("SET dict:", merge_result[1])

## 5. Informatica-Style Functions

### 5.1 infa_router — split DataFrame by condition

In [ ]:
# infa_router: takes a LIST of condition strings, returns a LIST of DataFrames
conditions = ["dept = 'Engineering'", "dept = 'Finance'", "dept NOT IN ('Engineering','Finance')"]
routed = infa_router(source_df, conditions)

print("Engineering group:")
display(routed[0])

### 5.2 infa_joiner

In [ ]:
dept_data = spark.createDataFrame([
    Row(dept="Engineering", budget=500000),
    Row(dept="Finance",     budget=300000),
    Row(dept="HR",          budget=150000),
])

# infa_joiner: args are (left_df, right_df, join_type, join_keys as list)
joined_df = infa_joiner(source_df, dept_data, join_type='left', join_keys=['dept'])
display(joined_df)

### 5.3 md5_checksum

In [ ]:
# md5_checksum: args are (df_md5, column_list, checksum_name)
checksum_df = md5_checksum(source_df, ['first_name', 'last_name', 'salary'], 'row_hash')
display(checksum_df.select('id', 'first_name', 'row_hash'))

### 5.4 generate_insert_update_delete_flags

Compares source and target DataFrames and tags each row with I (insert), U (update), or D (delete).

In [ ]:
# generate_insert_update_delete_flags:
# Requires a pre-joined DataFrame (source outer-joined with target) with checksums from both sides.
# Step 1: add checksums to source and target
src_cs = md5_checksum(source_df, ['first_name', 'last_name', 'salary', 'dept'], 'src_checksum')
tgt_cs = md5_checksum(target_df, ['first_name', 'last_name', 'salary', 'dept'], 'tgt_checksum')

from pyspark.sql.functions import col as F_col
tgt_renamed = tgt_cs.select(F_col('id').alias('tgt_id'), F_col('tgt_checksum'))

# Step 2: full outer join on PK
joined = src_cs.join(tgt_renamed, src_cs.id == tgt_renamed.tgt_id, 'full')
joined = joined.withColumn('change_flag', lit('N'))

# Step 3: generate flags
flag_dict = {
    'src_key_column':    'id',
    'tgt_key_column':    'tgt_id',
    'src_checksum_name': 'src_checksum',
    'tgt_checksum_name': 'tgt_checksum',
    'tgt_change_flag':   'change_flag',
    'flag_names':        ['insert_flag', 'update_flag', 'delete_flag'],
    'flag_values':       [1, 0]
}
flagged_df = generate_insert_update_delete_flags(joined, flag_dict)
display(flagged_df.select('id', 'tgt_id', 'insert_flag', 'update_flag', 'delete_flag'))

### 5.5 generate_insert_update_delete_dataframes

Returns a dict of `{"insert": df, "update": df, "delete": df}`.

In [ ]:
# generate_insert_update_delete_dataframes: takes the flagged_df from above + a dictionary
df_dict = {
    'src_columns':    ['id', 'first_name', 'last_name', 'salary', 'dept'],
    'src_key_column': 'id',
    'tgt_key_column': 'tgt_id',
    'flag_names':     ['insert_flag', 'update_flag', 'delete_flag'],
    'flag_values':    [1, 0]
}
df_list = generate_insert_update_delete_dataframes(flagged_df, df_dict)
print(f"Inserts: {df_list[0].count()}"); display(df_list[0])

### 5.6 lookup_dataframe

In [ ]:
# lookup_dataframe: src_key_columns and lkp_key_columns must have DIFFERENT names
lookup_ref = spark.createDataFrame([
    Row(dept_lkp="Engineering", dept_code="ENG"),
    Row(dept_lkp="Finance",     dept_code="FIN"),
    Row(dept_lkp="HR",          dept_code="HR"),
])

enriched_df = lookup_dataframe(
    df=source_df,
    df_lkp=lookup_ref,
    src_key_columns=['dept'],
    lkp_key_columns=['dept_lkp'],   # different name from src key
    lkp_return_columns=['dept_code']
)
display(enriched_df)

## 6. Regression Testing Functions

### 6.1 compare_schemas

In [ ]:
# Same schema — expect no differences
result = compare_schemas(source_df, target_df)
print("Schema comparison result:", result)

# Add an extra column to one DF to trigger a mismatch
from pyspark.sql.functions import lit
modified_df = source_df.withColumn("extra_col", lit("test"))
result_mismatch = compare_schemas(source_df, modified_df)
print("Schema mismatch result:", result_mismatch)

### 6.2 hash_comp — row-level hash comparison

In [ ]:
# hash_comp: just two DataFrames, no pk_columns argument — hashes entire DF
print("Same DataFrame:")
hash_comp(source_df, source_df)
print("\nDifferent DataFrames:")
hash_comp(source_df, target_df)

### 6.3 compare_dataframes — full diff

In [ ]:
# compare_dataframes: prints differences, returns None
compare_dataframes(source_df, target_df)

### 6.4 check_for_special_characters

In [ ]:
# check_for_special_characters: takes just a DataFrame, checks all string columns
special_data = spark.createDataFrame([
    Row(id=1, name="Alice & Bob"),
    Row(id=2, name="Carol <normal>"),
    Row(id=3, name="Dave"),
])
check_for_special_characters(special_data)

## 7. DML Operation Helper — dml_operation_df

Builds an insert/update/delete DML statement string from a DataFrame row.

In [ ]:
# dml_operation_df: args are (source_df, target_df, pk_columns, mode)
# mode = 'insert' | 'update' | 'delete'
insert_df = dml_operation_df(source_df, target_df, pk_columns=['id'], mode='insert')
print("Records to INSERT:"); display(insert_df)

## 8. Utility Helpers

### 8.1 julian_to_timestamp / epoch_to_datetime

In [ ]:
# julian_to_timestamp / epoch_to_datetime: take single scalar values, return formatted strings
print("Julian 2460000 →", julian_to_timestamp(2460000))
print("Epoch 1700000000 →", epoch_to_datetime(1700000000))

### 8.2 remove_null_from_dictionary

In [ ]:
# remove_null_from_dictionary: takes a LIST of dicts, removes None values from each
records = [
    {'id': 1, 'name': 'Alice', 'dept': None},
    {'id': 2, 'name': None,   'dept': 'Finance'},
    {'id': 3, 'name': 'Carol', 'dept': 'Engineering'},
]
clean = remove_null_from_dictionary(records)
for r in clean:
    print(r)

### 8.3 column_clear — fill column with a default value

In [ ]:
# column_clear: clears a single column to NULL (no fill_value, no list)
cleared_df = column_clear(source_df, 'salary')
display(cleared_df)

---
## Summary

| Section | Functions Tested |
|---------|------------------|
| Global Utilities | GetTime, col_rename, uppercase_columns, trim_column_values, add_column_prefix, to_string_datatype, calculate_df_size, GetValueFromDataframe, find_and_replace, truncate_dataframe, generate_update_setString, generate_delta_merge_conditions |
| Informatica Functions | infa_router, infa_joiner, md5_checksum, generate_insert_update_delete_flags, generate_insert_update_delete_dataframes, lookup_dataframe |
| Regression Testing | compare_schemas, hash_comp, compare_dataframes, check_for_special_characters |
| DML Helpers | dml_operation_df |
| Utilities | julian_to_timestamp, epoch_to_datetime, remove_null_from_dictionary, column_clear |

For cloud-dependent functions (blob, snowflake, on-prem JDBC, Salesforce, REST API),
refer to `notes/framework_progress.md` for the setup roadmap.